## 21B. LightGBM без RD: оффлайн-обучение на 18–28 февраля и forward-тест 1–4 марта 2026

**Идея:** жёстко разделяем две задачи:

1. \(\) **Скачивание сырых 1m-свечей с Bybit в локальную папку** `data/test_bybit_1m` (по одной монете и одному дню).
2. \(\) **Оффлайн-обучение и forward-тест**, которые работают *только* с локальными файлами и больше не ходят в Bybit API.

Так мы избегаем rate-limit, можем наполнять датасет кусками, а все эксперименты становятся повторяемыми и детерминированными.

**Цель эксперимента:**
- Обучить LightGBM **без RD-фичей** на окне 18–28 февраля 2026 (10 train-дней + 1 val-день) по тем же топ-50 монетам, что и в ноутбуке 20.
- Проверить forward-результаты на 1–4 марта 2026 по тем же монетам и той же логике бэктеста `signal_flip` (0.75, комиссия 0.1%).
- Сравнить с ноутбуком 20, где модель без RD обучалась на 1–8/9/10 февраля.

In [1]:
import sys, os, numpy as np, pandas as pd, warnings, datetime as dt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == '05_experiments' else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.features.warmup_loader import _fetch_klines_from_bybit, _to_bybit_symbol

COMMISSION_RT = 0.001
THRESHOLD = 0.75  # band 25-75
TARGET_COL = 'target'

# Папка для сырых 1m-данных Bybit (один файл = одна монета + один день)
TEST_DATA_DIR = os.path.join(_root, 'data', 'test_bybit_1m')
os.makedirs(TEST_DATA_DIR, exist_ok=True)

print('Root:', _root)
print('TEST_DATA_DIR:', TEST_DATA_DIR)

Root: c:\project\trading_bot_2Engine
TEST_DATA_DIR: c:\project\trading_bot_2Engine\data\test_bybit_1m


### 1. Базовые фичи, набор без RD и топ-50 монет (как в ноутбуке 20)

Здесь мы только:
- читаем исходный parquet `data_labeled_tp_sl_1_05.parquet` и список `BASE_FEATURES`,
- строим `BASE_NO_RD` и `KEY_FEATS_NO_RD`,
- пересчитываем `top50_symbols` по длине сессий на тестовом дне 10.02.2026 — как в ноутбуке 20, чтобы результаты были сопоставимы.

In [2]:
labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path    = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')

df_raw = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_raw.columns]

df_tmp = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
df_tmp = df_tmp[df_tmp[TARGET_COL].isin([-1.0, 1.0])]
df_tmp['y'] = (df_tmp[TARGET_COL] == 1).astype(int)
df_tmp['date'] = pd.to_datetime(df_tmp['datetime'], utc=True).dt.date

dates = sorted(df_tmp['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней в parquet, найдено {len(dates)}'

test_date = dates[9]  # 10-й день = 2026-02-10
df_test_day = df_tmp[df_tmp['date'] == test_date].copy()
sym_counts = df_test_day.groupby('symbol').size().sort_values(ascending=False)
top50_symbols = sym_counts.head(50).index.tolist()

def is_rd_feature(c: str) -> bool:
    if c.startswith('rd_'):
        return True
    if c == 'abs_rd':
        return True
    return False

BASE_NO_RD = [c for c in BASE_FEATURES if not is_rd_feature(c)]
KEY_FEATS_NO_RD = [c for c in BASE_NO_RD if c in ['ret_1', 'ret_5', 'rsi_14', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position']]
SEQ_WINDOWS = [5, 15, 30, 60]

print('BASE_FEATURES (всего):', len(BASE_FEATURES))
print('BASE_NO_RD (без RD):', len(BASE_NO_RD), BASE_NO_RD)
print('KEY_FEATS_NO_RD:', KEY_FEATS_NO_RD)
print('Top 50 symbols by rows on test day:', len(top50_symbols))
print('Sample symbols:', top50_symbols[:10])

BASE_FEATURES (всего): 22
BASE_NO_RD (без RD): 13 ['ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
KEY_FEATS_NO_RD: ['ret_1', 'ret_5', 'rsi_14', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position']
Top 50 symbols by rows on test day: 50
Sample symbols: ['WHITEWHALE', 'POWER', 'PIPPIN', 'GPS', 'ROAM', 'ZKP', 'BEAT', 'STABLE', 'XNY', 'RIVER']


### 2. Функции бэктеста и построения фичей без RD (как в 18/19/20)

Здесь нет никаких запросов к Bybit — только арифметика по OHLCV и времени.

In [3]:
def backtest_prod_hold(proba, ret, session_ids, threshold=THRESHOLD, commission_rt=COMMISSION_RT):
    """signal_flip: BUY >= thr, SELL <= 1-thr, HOLD держим позицию."""
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt_ = pd.to_datetime(df['datetime'], utc=True)
    hour = dt_.dt.hour + dt_.dt.minute / 60
    dow = dt_.dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df['dow_cos'] = np.cos(2 * np.pi * dow / 7)
    return df

def add_ohlc_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rng = df['high'] - df['low']
    rng = rng.replace(0, np.nan)
    body = (df['close_price'] - df['open']).abs()
    df['body_ratio'] = (body / rng).fillna(0.5).clip(0, 1)
    df['close_position'] = ((df['close_price'] - df['low']) / rng).fillna(0.5).clip(0, 1)
    return df

def add_price_volume_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    grp = df.groupby('session_key', group_keys=False)
    close = grp['close_price']
    df['ret_1'] = close.pct_change(1)
    df['ret_5'] = close.pct_change(5)
    # RSI(14)
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.rolling(14, min_periods=1).mean()
    avg_loss = loss.rolling(14, min_periods=1).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan).fillna(1e-9))
    df['rsi_14'] = 100 - (100 / (1 + rs))
    # MACD(12,26,9)
    ema12 = close.transform(lambda x: x.ewm(span=12, adjust=False).mean())
    ema26 = close.transform(lambda x: x.ewm(span=26, adjust=False).mean())
    df['macd_line'] = ema12 - ema26
    df['macd_signal'] = df.groupby('session_key', group_keys=False)['macd_line'].transform(
        lambda x: x.ewm(span=9, adjust=False).mean()
    )
    df['macd_hist'] = df['macd_line'] - df['macd_signal']
    # volatility_14
    df['volatility_14'] = close.pct_change().rolling(14, min_periods=1).std()
    # volume_rel_20
    vol_ma = grp['volume'].transform(lambda x: x.rolling(20, min_periods=1).mean())
    df['volume_rel_20'] = df['volume'] / (vol_ma + 1e-9)
    return df

def build_features_no_rd(df: pd.DataFrame) -> pd.DataFrame:
    df = add_time_features(df)
    df = add_ohlc_features(df)
    df = add_price_volume_features(df)
    grp = df.groupby('session_key', group_keys=False)
    for w in SEQ_WINDOWS:
        for c in KEY_FEATS_NO_RD:
            df[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f'{c}_roll{w}_std']  = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
    return df

### 3. Функции для скачивания и локальной загрузки 1m-свечей Bybit

- `download_bybit_day` — **единственное место**, где мы ходим в Bybit API.
- `load_bybit_day` — читает уже сохранённый `parquet` из `TEST_DATA_DIR` и никогда не трогает API.

Скачивание можно выполнять батчами, отдельными ячейками, чтобы не ловить rate limit.

In [4]:
# Хелпер: докачать (или перекачать) полный день, если в файле только ~1000 баров

def download_or_complete_day(symbol: str, date: dt.date, target_rows: int = 1440) -> pd.DataFrame:
    """Гарантирует максимально полный день 1m-свечей в TEST_DATA_DIR для symbol+date.

    - Если файл уже есть и строк >= target_rows-10, просто читаем и возвращаем.
    - Если файл есть, но строк заметно меньше (например, 1000), перекачиваем полный день с нуля.
    - Если файла нет, качаем полный день с нуля.

    Для скачивания полного дня используем пагинацию по 1000 свечей назад от конца дня.
    """
    filename = f"{symbol}_{date.isoformat()}.parquet"
    path = os.path.join(TEST_DATA_DIR, filename)

    if os.path.exists(path):
        df_existing = pd.read_parquet(path)
        n = len(df_existing)
        print(f"Файл уже есть: {filename}, строк={n}")
        if n >= target_rows - 10:
            # считаем, что день уже достаточно полный
            print("День считается полным, докачка не требуется")
            return df_existing.sort_values("datetime").reset_index(drop=True) if "datetime" in df_existing.columns else df_existing
        else:
            print("Файл есть, но день неполный — перекачиваем целиком")

    # Полная перекачка дня с нуля (пагинация по 1000 баров)
    symbol_bybit = _to_bybit_symbol(symbol)
    end_dt = dt.datetime(date.year, date.month, date.day, 23, 59, 59, tzinfo=dt.timezone.utc)
    end_ms = int(end_dt.timestamp() * 1000)

    frames = []
    total_rows = 0
    max_chunks = 3  # 3*1000 достаточно, чтобы покрыть 1440 минут

    for chunk_idx in range(max_chunks):
        print(f"Запрос Bybit [{chunk_idx+1}/{max_chunks}]: symbol={symbol_bybit}, end={end_dt}, limit=1000")
        df_chunk = _fetch_klines_from_bybit(symbol_bybit, end_ts_ms=end_ms, limit=1000)
        if df_chunk.empty:
            print("Пустой ответ от Bybit, останавливаем пагинацию")
            break

        df_chunk['datetime'] = pd.to_datetime(df_chunk['timestamp'], unit='ms', utc=True)
        df_chunk['date'] = df_chunk['datetime'].dt.date
        df_chunk = df_chunk[df_chunk['date'] == date].copy()
        if df_chunk.empty:
            print("В чанке нет строк целевого дня, останавливаем пагинацию")
            break

        frames.append(df_chunk)
        total_rows += len(df_chunk)
        print(f"  чанк {chunk_idx+1}: строк целевого дня = {len(df_chunk)}, всего накоплено = {total_rows}")

        earliest_ts = df_chunk['timestamp'].min()
        end_ms = int(earliest_ts - 60_000)
        end_dt = dt.datetime.utcfromtimestamp(end_ms / 1000.0).replace(tzinfo=dt.timezone.utc)

        if total_rows >= target_rows:
            break

    if not frames:
        print("Не удалось получить данные для этого дня, файл не сохранён")
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True).sort_values('timestamp').reset_index(drop=True)
    df['symbol'] = symbol

    try:
        df.to_parquet(path)
        print(f"Сохранено: {path}, строк: {len(df)}")
    except Exception as e:
        print(f"Не удалось сохранить {path}: {e}")

    return df

In [5]:
# Переопределяем download_bybit_day, чтобы закачивать полный день (~1440 минут) через несколько запросов по 1000 свечей

def download_bybit_day(symbol: str, date: dt.date, limit: int = 1500, overwrite: bool = False) -> pd.DataFrame:
    """Скачивает максимально полный день 1m Kline по символу и дате из Bybit V5 и сохраняет в TEST_DATA_DIR.

    Один файл = одна монета + один день, имя: SYMBOL_YYYY-MM-DD.parquet.
    Bybit за один запрос даёт максимум 1000 свечей, поэтому делаем
    несколько запросов по 1000, пока не доберём хотя бы ~1440 минут текущего дня
    (или не упрёмся в пустые ответы).
    """
    filename = f"{symbol}_{date.isoformat()}.parquet"
    path = os.path.join(TEST_DATA_DIR, filename)

    if os.path.exists(path) and not overwrite:
        print(f"Уже есть: {filename}, overwrite=False — читаю из файла")
        return pd.read_parquet(path)

    symbol_bybit = _to_bybit_symbol(symbol)
    end_dt = dt.datetime(date.year, date.month, date.day, 23, 59, 59, tzinfo=dt.timezone.utc)
    end_ms = int(end_dt.timestamp() * 1000)

    frames = []
    total_rows = 0
    max_chunks = 3  # 3 * 1000 достаточно, чтобы покрыть 1440 минут

    for chunk_idx in range(max_chunks):
        print(f"Запрос Bybit [{chunk_idx+1}/{max_chunks}]: symbol={symbol_bybit}, end={end_dt}, limit=1000")
        df_chunk = _fetch_klines_from_bybit(symbol_bybit, end_ts_ms=end_ms, limit=1000)
        if df_chunk.empty:
            print("Пустой ответ от Bybit, останавливаем пагинацию")
            break

        df_chunk['datetime'] = pd.to_datetime(df_chunk['timestamp'], unit='ms', utc=True)
        df_chunk['date'] = df_chunk['datetime'].dt.date
        df_chunk = df_chunk[df_chunk['date'] == date].copy()
        if df_chunk.empty:
            print("В чанке нет строк целевого дня, останавливаем пагинацию")
            break

        frames.append(df_chunk)
        total_rows += len(df_chunk)
        print(f"  чанк {chunk_idx+1}: строк целевого дня = {len(df_chunk)}, всего накоплено = {total_rows}")

        # следующий end_ms = самая ранняя свеча текущего чанка - 1 минута
        earliest_ts = df_chunk['timestamp'].min()
        end_ms = int(earliest_ts - 60_000)
        end_dt = dt.datetime.utcfromtimestamp(end_ms / 1000.0).replace(tzinfo=dt.timezone.utc)

        if total_rows >= 1440:
            break

    if not frames:
        print("Не удалось получить данные для этого дня, файл не сохранён")
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True).sort_values('timestamp').reset_index(drop=True)
    df['symbol'] = symbol

    try:
        df.to_parquet(path)
        print(f"Сохранено: {path}, строк: {len(df)}")
    except Exception as e:
        print(f"Не удалось сохранить {path}: {e}")

    return df


In [6]:
def download_bybit_day(symbol: str, date: dt.date, limit: int = 1500, overwrite: bool = False) -> pd.DataFrame:
    """Скачивает 1m Kline по символу и дате из Bybit V5 и сохраняет в TEST_DATA_DIR.

    Один файл = одна монета + один день, имя: SYMBOL_YYYY-MM-DD.parquet.
    Используется для поштучного наполнения локального тестового датасета.
    """
    filename = f"{symbol}_{date.isoformat()}.parquet"
    path = os.path.join(TEST_DATA_DIR, filename)

    if os.path.exists(path) and not overwrite:
        print(f"Уже есть: {filename}, overwrite=False — читаю из файла")
        return pd.read_parquet(path)

    symbol_bybit = _to_bybit_symbol(symbol)
    end_dt = dt.datetime(date.year, date.month, date.day, 23, 59, 59, tzinfo=dt.timezone.utc)
    end_ms = int(end_dt.timestamp() * 1000)

    print(f"Запрос Bybit: symbol={symbol_bybit}, date={date}, limit={limit}")
    df = _fetch_klines_from_bybit(symbol_bybit, end_ts_ms=end_ms, limit=limit)
    if df.empty:
        print("Пустой ответ от Bybit, файл не сохранён")
        return df

    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df['date'] = df['datetime'].dt.date
    df = df[df['date'] == date].copy()
    df['symbol'] = symbol
    df = df.sort_values('datetime').reset_index(drop=True)

    try:
        df.to_parquet(path)
        print(f"Сохранено: {path}, строк: {len(df)}")
    except Exception as e:
        print(f"Не удалось сохранить {path}: {e}")

    return df

def load_bybit_day(symbol: str, date: dt.date) -> pd.DataFrame:
    """Загружает 1m Kline по символу за указанный день из локальной папки TEST_DATA_DIR.

    Важно: эта функция НЕ ходит в Bybit API.
    Перед запуском обучения/forward-теста данные должны быть заранее
    скачаны через `download_bybit_day`.
    """
    filename = f"{symbol}_{date.isoformat()}.parquet"
    path = os.path.join(TEST_DATA_DIR, filename)
    if not os.path.exists(path):
        return pd.DataFrame()
    df = pd.read_parquet(path)
    if 'datetime' in df.columns:
        df = df.sort_values('datetime').reset_index(drop=True)
    return df

### 3A. Примеры ячеек для скачивания данных (запускать вручную)

Эти ячейки можно запускать по мере необходимости. Они **наполняют TEST_DATA_DIR**, после чего блоки обучения и forward-теста
будут использовать только локальные файлы.

In [7]:
# Пример: скачать 18–28 февраля 2026 для одной монеты
for d in range(18, 29):
     _ = download_bybit_day('WHITEWHALE', dt.date(2026, 2, d))

# Пример: скачать 18–28 февраля 2026 для всех top-50 монет (лучше делать частями)
train_val_dates = [dt.date(2026, 2, d) for d in range(18, 29)]
for d in train_val_dates:
    for sym in top50_symbols:
        _ = download_bybit_day(sym, d)

# Пример: скачать 1–4 марта 2026 для всех top-50 монет
forward_dates = [dt.date(2026, 3, d) for d in [1, 2, 3, 4]]
for d in forward_dates:
    for sym in top50_symbols:
        _ = download_bybit_day(sym, d)

Уже есть: WHITEWHALE_2026-02-18.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-19.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-20.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-21.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-22.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-23.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-24.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-25.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-26.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-27.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-28.parquet, overwrite=False — читаю из файла
Уже есть: WHITEWHALE_2026-02-18.parquet, overwrite=False — читаю из файла
Уже есть: POWER_2026-02-18.parquet, overwrite=False — читаю из файла
Уже есть: PIPPIN_2026-02-18.parquet, overwr

### 3C. Проверка существующих файлов и докачка до полного дня

Этот блок:
- пробегается по всем `SYMBOL_YYYY-MM-DD.parquet` в `TEST_DATA_DIR`,
- считает, сколько там строк,
- показывает список файлов, у которых строк **заметно меньше, чем у полного дня** (по умолчанию `< 1400`),
- по флагу `RUN_FIX` может **докачать/перекачать** только эти неполные дни через `download_or_complete_day`.

Так можно постепенно привести все файлы к полному дню без лишних запросов к Bybit.

In [ ]:
# Проверка и опциональная докачка уже существующих файлов до полного дня

from pathlib import Path

TARGET_ROWS = 1440
RUN_FIX = False    # выставь True, чтобы реально докачать неполные дни
MAX_FIX = 10       # максимум файлов для докачки за один запуск

files = [f for f in os.listdir(TEST_DATA_DIR) if f.endswith('.parquet')]
rows_info = []
for fname in files:
    path = os.path.join(TEST_DATA_DIR, fname)
    try:
        n = len(pd.read_parquet(path))
    except Exception as e:
        print(f"Не удалось прочитать {fname}: {e}")
        continue
    rows_info.append((fname, n))

rows_df = pd.DataFrame(rows_info, columns=['filename', 'rows']).sort_values('rows')
print("Всего файлов в TEST_DATA_DIR:", len(rows_df))
print("Статистика по строкам:")
print(rows_df['rows'].describe())

# Неполные дни — сильно меньше целевого количества строк
incomplete = rows_df[rows_df['rows'] < TARGET_ROWS - 10].copy()
print("Неполных файлов (rows < TARGET_ROWS-10):", len(incomplete))
print("Первые 10 неполных:")
print(incomplete.head(10))

if RUN_FIX and not incomplete.empty:
    to_fix = incomplete.head(MAX_FIX)
    print(f"\nДокачиваем/перекачиваем максимум {len(to_fix)} файлов:\n")
    for fname, n in to_fix.itertuples(index=False):
        # вытаскиваем symbol и date из имени файла SYMBOL_YYYY-MM-DD.parquet
        stem = Path(fname).stem
        symbol, date_str = stem.split('_', 1)
        d = dt.date.fromisoformat(date_str)
        print(f"Фиксим {fname} (rows={n})...")
        _ = download_or_complete_day(symbol, d, target_rows=TARGET_ROWS)
    print("Докачка неполных файлов завершена.")
else:
    print("RUN_FIX=False или неполных файлов нет — только показана статистика, докачка не выполнялась.")

In [8]:
# Докачка недостающих файлов по top-50 монетам

from pathlib import Path

# Дни для обучения (18–28 февраля) и forward (1–4 марта)
train_val_dates = [dt.date(2026, 2, d) for d in range(18, 29)]
forward_dates = [dt.date(2026, 3, d) for d in [1, 2, 3, 4]]

expected = []
for d in train_val_dates + forward_dates:
    for sym in top50_symbols:
        expected.append((sym, d))

# Какие файлы уже есть
existing_files = {Path(p).name for p in os.listdir(TEST_DATA_DIR)}

missing = []
for sym, d in expected:
    fname = f"{sym}_{d.isoformat()}.parquet"
    if fname not in existing_files:
        missing.append((sym, d, fname))

print(f"Всего ожидаемых файлов: {len(expected)}")
print(f"Уже есть файлов: {len(existing_files)}")
print(f"Не хватает файлов: {len(missing)}")
print("Первые 10 отсутствующих:")
for row in missing[:10]:
    print(row)

# ВНИМАНИЕ: этот цикл реально ходит в Bybit и может зависать из-за rate limit.
# По умолчанию не запускаем скачивание автоматически — сначала смотрим список missing.
RUN_DOWNLOAD = False   # выставь True вручную, когда захочешь докачать партию
first_n = 10           # при необходимости уменьшить/увеличить

if RUN_DOWNLOAD and missing:
    for i, (sym, d, fname) in enumerate(missing[:first_n], start=1):
        print(f"[{i}/{min(first_n, len(missing))}] докачиваем {fname}")
        _ = download_bybit_day(sym, d)

    print("Докачка партии завершена.")
else:
    print("Скачивание не запущено (RUN_DOWNLOAD=False). Только показан список недостающих файлов.")

Всего ожидаемых файлов: 750
Уже есть файлов: 735
Не хватает файлов: 15
Первые 10 отсутствующих:
('SAROS', datetime.date(2026, 2, 18), 'SAROS_2026-02-18.parquet')
('SAROS', datetime.date(2026, 2, 19), 'SAROS_2026-02-19.parquet')
('SAROS', datetime.date(2026, 2, 20), 'SAROS_2026-02-20.parquet')
('SAROS', datetime.date(2026, 2, 21), 'SAROS_2026-02-21.parquet')
('SAROS', datetime.date(2026, 2, 22), 'SAROS_2026-02-22.parquet')
('SAROS', datetime.date(2026, 2, 23), 'SAROS_2026-02-23.parquet')
('SAROS', datetime.date(2026, 2, 24), 'SAROS_2026-02-24.parquet')
('SAROS', datetime.date(2026, 2, 25), 'SAROS_2026-02-25.parquet')
('SAROS', datetime.date(2026, 2, 26), 'SAROS_2026-02-26.parquet')
('SAROS', datetime.date(2026, 2, 27), 'SAROS_2026-02-27.parquet')
Скачивание не запущено (RUN_DOWNLOAD=False). Только показан список недостающих файлов.


### 4. Построение train/val на 18–28 февраля (оффлайн, только локальные файлы)

Предполагаем, что все необходимые файлы уже есть в `TEST_DATA_DIR`:
- `SYMBOL_2026-02-18.parquet` ... `SYMBOL_2026-02-28.parquet` для всех монет из `top50_symbols`.

In [9]:
train_val_dates = [dt.date(2026, 2, d) for d in range(18, 29)]  # 18–28 февраля
train_dates_feb = train_val_dates[:-1]  # 10 дней: 18–27
val_date_feb = train_val_dates[-1]      # 28 февраля

frames = []
for d in train_val_dates:
    for sym in top50_symbols:
        df_sym = load_bybit_day(sym, d)
        if df_sym.empty:
            continue
        df_sym['session_key'] = df_sym['symbol'].astype(str) + '_' + df_sym['date'].astype(str)
        frames.append(df_sym)

if not frames:
    raise RuntimeError('Нет локальных файлов в TEST_DATA_DIR для 18–28 февраля по top-50 символам')

df_feb = pd.concat(frames, ignore_index=True).sort_values(['session_key', 'datetime']).reset_index(drop=True)

# Подсветим длины сессий
sess_len = df_feb.groupby('session_key').size()
print('Всего сессий (18–28 фев):', len(sess_len))
print('Мин/медиана/макс длины сессий:', sess_len.min(), sess_len.median(), sess_len.max())
short_sess = sess_len[sess_len < 1000]
print('Коротких сессий (<1000 баров):', len(short_sess))
if len(short_sess) > 0:
    print(short_sess.head())

# Строим ret_next и фичи
df_feb['ret_next'] = df_feb.groupby('session_key')['close_price'].pct_change().shift(-1)
df_feb = df_feb.dropna(subset=['ret_next']).reset_index(drop=True)
df_feb_feat = build_features_no_rd(df_feb)

FEATURES_SEQ_NO_RD = BASE_NO_RD + [
    f'{c}_roll{w}_{s}'
    for w in SEQ_WINDOWS
    for c in KEY_FEATS_NO_RD
    for s in ('mean', 'std')
]

cols = [c for c in FEATURES_SEQ_NO_RD if c in df_feb_feat.columns]
df_feb_feat = df_feb_feat.dropna(subset=cols + ['ret_next']).copy()

print('FEATURES_SEQ_NO_RD (фактически использовано):', len(cols))
print('Всего строк после фичей (18–28 фев):', len(df_feb_feat))

# Разделяем по датам: train 18–27, val 28 февраля
train_mask = df_feb_feat['date'].isin(train_dates_feb)
val_mask = df_feb_feat['date'] == val_date_feb

train_df = df_feb_feat[train_mask].copy()
val_df = df_feb_feat[val_mask].copy()

print('Train dates:', sorted(train_dates_feb))
print('Val date:', val_date_feb)
print('Rows train/val:', len(train_df), len(val_df))

scaler_no_rd = StandardScaler()
X_tr_no_rd = scaler_no_rd.fit_transform(train_df[cols].fillna(0))
X_va_no_rd = scaler_no_rd.transform(val_df[cols].fillna(0))
y_tr = (train_df['ret_next'] > 0).astype(int).values
y_va = (val_df['ret_next'] > 0).astype(int).values

lgb_no_rd = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)
lgb_no_rd.fit(X_tr_no_rd, y_tr)
p_va = lgb_no_rd.predict_proba(X_va_no_rd)[:, 1]
print('AUC no-RD на валидации (28 фев):', roc_auc_score(y_va, p_va))

Всего сессий (18–28 фев): 539
Мин/медиана/макс длины сессий: 1000 1000.0 1000
Коротких сессий (<1000 баров): 0
FEATURES_SEQ_NO_RD (фактически использовано): 69
Всего строк после фичей (18–28 фев): 535766
Train dates: [datetime.date(2026, 2, 18), datetime.date(2026, 2, 19), datetime.date(2026, 2, 20), datetime.date(2026, 2, 21), datetime.date(2026, 2, 22), datetime.date(2026, 2, 23), datetime.date(2026, 2, 24), datetime.date(2026, 2, 25), datetime.date(2026, 2, 26), datetime.date(2026, 2, 27)]
Val date: 2026-02-28
Rows train/val: 487060 48706
AUC no-RD на валидации (28 фев): 0.5945995658863434


### 5. Forward‑тест 1–4 марта 2026 по тем же 50 символам (оффлайн)

Здесь также предполагается, что файлы `SYMBOL_2026-03-01.parquet` ... `SYMBOL_2026-03-04.parquet` уже лежат в `TEST_DATA_DIR`.

In [10]:
forward_dates = [dt.date(2026, 3, d) for d in [1, 2, 3, 4]]

def forward_test_day(target_date: dt.date) -> dict:
    frames = []
    for sym in top50_symbols:
        df_sym = load_bybit_day(sym, target_date)
        if df_sym.empty:
            continue
        df_sym['session_key'] = df_sym['symbol'].astype(str) + '_' + df_sym['date'].astype(str)
        frames.append(df_sym)
    if not frames:
        return {'date': target_date, 'rows': 0, 'net_%': np.nan, 'trades': 0, 'avg_%_per_trade': np.nan}

    df_day = pd.concat(frames, ignore_index=True).sort_values(['session_key', 'datetime']).reset_index(drop=True)
    sess_len = df_day.groupby('session_key').size()
    print(f'[{target_date}] сессий:', len(sess_len), 'мин/медиана/макс:', sess_len.min(), sess_len.median(), sess_len.max())

    df_day['ret_next'] = df_day.groupby('session_key')['close_price'].pct_change().shift(-1)
    df_day = df_day.dropna(subset=['ret_next']).reset_index(drop=True)

    df_feat = build_features_no_rd(df_day)
    cols_local = [c for c in FEATURES_SEQ_NO_RD if c in df_feat.columns]
    X = scaler_no_rd.transform(df_feat[cols_local].fillna(0))
    proba = lgb_no_rd.predict_proba(X)[:, 1]
    bt = backtest_prod_hold(proba, df_feat['ret_next'].values, df_feat['session_key'].values, threshold=THRESHOLD)
    return {
        'date': target_date,
        'rows': len(df_feat),
        'net_%': bt['net_%'],
        'trades': bt['trades'],
        'avg_%_per_trade': bt['avg_%_per_trade'],
    }

forward_results = [forward_test_day(d) for d in forward_dates]
forward_df = pd.DataFrame(forward_results)
forward_df

[2026-03-01] сессий: 49 мин/медиана/макс: 1000 1000.0 1000
[2026-03-02] сессий: 49 мин/медиана/макс: 1000 1000.0 1000
[2026-03-03] сессий: 49 мин/медиана/макс: 1000 1000.0 1000
[2026-03-04] сессий: 49 мин/медиана/макс: 1000 1000.0 1000


,date,rows,net_%,trades,avg_%_per_trade
0,2026-03-01,48951,43.901717,27,1.625990
1,2026-03-02,48951,-90.790124,26,-3.491928
2,2026-03-03,48951,110.597493,30,3.686583
3,2026-03-04,48951,-62.178234,36,-1.727173


### 6. Интерпретация результатов (после выполнения)

После выполнения ноутбука нужно сравнить:
- `forward_df` из этого ноутбука (модель без RD, обученная на 18–28 февраля),
- и результаты forward-теста из ноутбука 20 (модель без RD, обученная на 1–8/9/10 февраля).

Главный вопрос: становится ли `avg_%_per_trade` по дням 1–4 марта более стабильным, если обучаться ближе к forward-периоду,
или дрейф рынка всё равно «ломает» логику модели без RD.